# OpenIE
Resolve references, then extract subjects, relations, objects, and qualifiers. Compare resolved sentences with their original source in DataFrames.

Run `uv sync` and select the project kernel. Set `run_live=True` to call the configured model. Offsets always refer to the original Markdown.

In [ ]:
from utils import sentence_rows, setup_notebook, to_dataframe, write_json

from IPython.display import display

from data_platform.digest.openie import extract_claims
from data_platform.digest.source import parse_document

project_root = setup_notebook()


## 1. Configure and preview

In [ ]:
markdown_doc = "# Report\n\n## Factory\n\nAcme builds cars. It plans buses in 2027."
model = "openai:gpt-6-luna"
run_live = False
max_concurrency = 20
document = parse_document(markdown_doc)
display(
    to_dataframe(
        sentence_rows(document),
        columns=["index", "text", "start", "end"],
    )
)


## 2. Extract and review

In [ ]:
result = None
if run_live:
    result = await extract_claims(
        markdown_doc,
        resolution_model=model,
        extraction_model=model,
        max_concurrency=max_concurrency,
    )

sentences_df = to_dataframe(
    [
        {
            "index": sentence.index,
            "original": sentence.text,
            "resolved": sentence.resolution.text,
            "status": sentence.status,
        }
        for sentence in (result.sentences if result is not None else [])
    ],
    columns=["index", "original", "resolved", "status"],
)
claims_df = to_dataframe(
    result.claims if result is not None else [],
    source=markdown_doc,
    columns=[
        "subject",
        "relation",
        "object",
        "qualifiers",
        "text",
        "source",
        "start",
        "end",
    ],
)
display(sentences_df)
display(claims_df)


## 3. Inspect resolution evidence
Replacement evidence is scoped to the sentence’s original section. For custom backend adapters, see the [OpenIE guide](../src/data_platform/digest/openie/README.md).

In [ ]:
replacements_df = to_dataframe(
    [
        {"sentence_index": sentence.index, **replacement.model_dump(mode="json")}
        for sentence in (result.sentences if result is not None else [])
        for replacement in sentence.resolution.replacements
    ],
    columns=["sentence_index", "original", "replacement", "evidence"],
)
display(replacements_df)


## 4. Export (optional)
Saving replaces the demo CSV and JSON outputs. JSON preserves nested qualifiers, evidence, and source text.

In [ ]:
save_output = False
if save_output and result is not None:
    output_dir = project_root / ".data/openie"
    output_dir.mkdir(parents=True, exist_ok=True)
    claims_df.to_csv(output_dir / "claims.csv", index=False)
    payload = {
        "source_markdown": markdown_doc,
        "result": result.model_dump(mode="json"),
    }
    write_json(output_dir / "result.json", payload)
